#Sentiment Analysis
Sentiment analysis is the name given to the process of classifying pieces of information (words, phrases, texts) into positive, negative and neutral.
It is possible for the classification to also contain extra information regarding how intense this sentiment is (e.g. not just positive/negative, but very positive, positive, negative, very negative) as well as nuanced emotions.

### Sentiment Lexicons
The most commonly used method for sentiment analysis that requires no prior data of the text (no labels for machine learning) is the lexicon method.

Lexicons are dictionaries that contain a collection of words and their corresponding sentiment value (for example ‘happy’ - positive, ‘sad’ - negative).
These lexicons are used for measuring the overall sentiment of a text by usually naively adding up all of these sentiment values. 


# Afinn
Afinn is one such lexicon that supports several languages (also works with emoticons)
https://github.com/fnielsen/afinn

In [ ]:
%%capture
%pip install afinn
%pip install matplotlib

In [ ]:
from matplotlib import pyplot as plt
from afinn import Afinn
afinn = Afinn()
print(afinn.score('thankful'))
print(afinn.score('I am thankful'))

In [ ]:
print(afinn.score('not thankful')) # Quite naive - score of ‘thankful’ and score of ‘not thankful’ are the same (positive 2)

In [ ]:
print(afinn.score('no thankful')) # (sum of 1 negative word and 1 positive word)
print(afinn.score('sad sad thankful')) # (sum of 2 negative words and 1 positive word)

In [ ]:
print(afinn.score('I am happy because it is sunny and warm'))
print(afinn.score("No, sorry, I can't do it because I'm too stressed"))

In [ ]:
afinn._dict #Taking a look at what’s under the hood

In [ ]:
print(len(afinn._dict)) # If we check the length of the dictionary, we can see that we have 3382 entries for positive/negative words

# Apply sentiment analysis to a book

In [ ]:
%%capture
%pip install gutenbergpy tidytext

In [ ]:
# We will try to apply sentiment analysis on Jane Austen’s book from last laboratory - Sense and Sensibility
import pandas as pd
import gutenbergpy.textget
from tidytext import unnest_tokens
import nltk
nltk.download('punkt')


def line_is_chapter(dataframe):
    chapter_list = []
    curr_chapter = 0
    for index, row in dataframe.iterrows():
        if re.search("^chapter [\\divxlc]*\.$", row['content'], re.IGNORECASE):
            curr_chapter += 1
        chapter_list.append(curr_chapter)
    return chapter_list



raw_book = gutenbergpy.textget.get_text_by_id(161)
sense_sensibility_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")
print(len(sense_sensibility_text)) # Length of our data

# Create our dataframe
import re
sense_sensibility_lines = sense_sensibility_text.splitlines()

sense_sensibility_df = pd.DataFrame({
    "content": sense_sensibility_lines,
    "line": list(range(len(sense_sensibility_lines)))
})

# The package provided us with data in a byte object format. Since we prefer working with strings, let's convert all content into utf-8 strings.
ss_df = sense_sensibility_df.copy()

curr_chapter = 0
sense_sensibility_df = sense_sensibility_df.assign(chapter = line_is_chapter(ss_df))
ss_df = ss_df.assign(chapter = line_is_chapter(ss_df)) # Split into chapters

ss_df = (unnest_tokens(ss_df, "word", "content"))
ss_df.reset_index(drop=True, inplace=True)
ss_df = ss_df[ss_df.word.notnull()].reset_index(drop=True)
ss_df


In [ ]:
print(afinn.score(sense_sensibility_text)) # Let’s see the overall sentiment of the book given by Afinn


### Exercise 1 
Find the afinn sentiment score for chapters 10, 30 and 50 of Jane Austen.
(print 3 separate values, 1 for each chapter)

In [ ]:
# Write code below
chapter_10_words = ' '.join(ss_df[ss_df.chapter == 10]['word'].to_list())
chapter_30_words = ' '.join(ss_df[ss_df.chapter == 30]['word'].to_list())
chapter_50_words = ' '.join(ss_df[ss_df.chapter == 50]['word'].to_list())

print(afinn.score(chapter_10_words))
print(afinn.score(chapter_30_words))
print(afinn.score(chapter_50_words))

In [ ]:
# Let's add all the word scores as information in the dataframe
word_scores = [afinn.score(word) for word in ss_df['word']]
word_scores

In [ ]:
ss_df = ss_df.assign(word_score = word_scores)
ss_df

In [ ]:
%%capture
%pip install siuba plotnine

In [ ]:
from siuba import * # Add this in each cell where you are using a siuba function
grouped_ss_df = group_by(ss_df, 'chapter') # Let's group the dataframe by chapter (which we kind of already have by the way we constructed our dataframe)
grouped_ss_df


In [ ]:
from siuba import *
summarized_ss_df = summarize(grouped_ss_df, chapter_score = _.word_score.sum()) # summarize the sentiment scores in each chapter
summarized_ss_df

In [ ]:
from plotnine import *

# We meet the "piping" format of writing code for the first time, ggplot likes to use this format for ease of writing
(ggplot(summarized_ss_df)         # defining what data to use
 + aes(x='chapter', y='chapter_score')    # defining what variables to use
 + geom_bar(stat='identity') # defining the type of plot to use
)


### Exercise 2
1. **Create 2 new dataframes called summarized_ss_df_pos and summarized_ss_df_neg**.

- summarized_ss_df_pos will contain the summarized chapter scores after **keeping only the positive words** (word_score => 0) from the initial dataframe.

- summarized_ss_df_neg will contain the summarized chapter scores after **keeping only the negative words** (word_score <= 0) from the initial dataframe.

    *chapter 1 for summarized_ss_df_pos should have 166 score, chapter 1 for summarized_ss_df_neg should have -67 score.*

2. **Plot the scores per chapter** (separately for each of the 2 dfs) of the new dataframes using ggplot.

In [ ]:
summarized_ss_df_pos = ss_df[ss_df.word_score > 0]
summarized_ss_df_pos = group_by(summarized_ss_df_pos, 'chapter')
summarized_ss_df_pos = summarize(summarized_ss_df_pos, chapter_score = _.word_score.sum())

summarized_ss_df_pos.head(n=30)

In [ ]:
(ggplot(summarized_ss_df_pos)         # defining what data to use
 + aes(x='chapter', y='chapter_score')    # defining what variables to use
 + geom_bar(stat='identity') # defining the type of plot to use
)

In [ ]:
summarized_ss_df_neg = ss_df[ss_df.word_score < 0]
summarized_ss_df_neg = group_by(summarized_ss_df_neg, 'chapter')
summarized_ss_df_neg = summarize(summarized_ss_df_neg, chapter_score = _.word_score.sum())

summarized_ss_df_neg.head(n=30)

In [ ]:
(ggplot(summarized_ss_df_neg)         # defining what data to use
 + aes(x='chapter', y='chapter_score')    # defining what variables to use
 + geom_bar(stat='identity') # defining the type of plot to use
)

# NRC
Nrc is another lexicon that contains somewhat different sentiment information. It categorizes each word into one or multiple of the following categories: anticipation, positive, trust, joy, surprise, negative, sadness, anger, disgust, fear

In [ ]:
%%capture
%pip install NRCLex

In [ ]:
# Applying it to Sense & Sensibility
from nrclex import NRCLex
text_object = NRCLex(sense_sensibility_text)
text_object.lexicon # Let's see what the words look like under the lexicon hood

In [ ]:
text_object.top_emotions # returns the strongest emotion(s)

In [ ]:
text_object.raw_emotion_scores # Word count per emotion

In [ ]:
text_object.affect_frequencies # Frequency of each emotion

In [ ]:
# We’ll add a new column with the top sentiment detected by NRC (part 1)
list_sentiments = [NRCLex(elem).top_emotions for elem in ss_df['word']]
print(list_sentiments[100])

list_first_sentiment = [elem[0] for elem in list_sentiments]
print(list_first_sentiment[100])


In [ ]:
# We’ll add a new column with the top sentiment detected by NRC (part 2)
ss_df_v2 = ss_df.copy()
ss_df_v2 = ss_df_v2.assign(sentiment = list_first_sentiment)

ss_df_v2[ss_df_v2.apply(lambda x: x['sentiment'][1] > 0, axis=1)] # filter all the entries for which the value is greater than 0
ss_df_v2

In [ ]:
# Let’s bring together the text per chapters
test_per_chapter = ss_df\
    .groupby('chapter')\
    .agg({
        'line': 'first', 
        'word': ', '.join, 
        'word_score': 'sum' 
    })\
    .reset_index() # get sum of all word_scores per chapter
test_per_chapter

In [ ]:
# Instead of adding any other information, let’s just do some simple visualizations
# Take all the occurrences of fear per chapter and plot them
# We'll use pyplot (plt) from matplotlib for this visualization to expand our visualizers toolbox

my_fear_list = [NRCLex(elem).raw_emotion_scores['fear'] if 'fear' in NRCLex(elem).raw_emotion_scores else 0 for elem in test_per_chapter['word']]
plt.plot(my_fear_list)
plt.xlabel('Chapter')
plt.ylabel('Fear occurrences')

### Exercise 3
Plot the graphs with the evolution of 4 sentiments of your choice (except fear) throughout the chapters of the book. Plot 1 graph for each of these 4 sentiments.

Inspect the graphs, compare them to one another and to the ones from afinn and see if you find any similarities or interesting observations. (e.g. a chapter with high anticipation seems to be followed by a chapter with high joy - example of a random observation).


In [ ]:
my_fear_list = [NRCLex(elem).raw_emotion_scores['disgust'] if 'disgust' in NRCLex(elem).raw_emotion_scores else 0 for elem in test_per_chapter['word']]
plt.plot(my_fear_list)
plt.xlabel('Chapter')
plt.ylabel('Disgust occurrences')

In [ ]:
my_fear_list = [NRCLex(elem).raw_emotion_scores['joy'] if 'joy' in NRCLex(elem).raw_emotion_scores else 0 for elem in test_per_chapter['word']]
plt.plot(my_fear_list)
plt.xlabel('Chapter')
plt.ylabel('Joy occurrences')

In [ ]:
my_fear_list = [NRCLex(elem).raw_emotion_scores['anticipation'] if 'anticipation' in NRCLex(elem).raw_emotion_scores else 0 for elem in test_per_chapter['word']]
plt.plot(my_fear_list)
plt.xlabel('Chapter')
plt.ylabel('Anticipation occurrences')

In [ ]:
my_fear_list = [NRCLex(elem).raw_emotion_scores['surprise'] if 'surprise' in NRCLex(elem).raw_emotion_scores else 0 for elem in test_per_chapter['word']]
plt.plot(my_fear_list)
plt.xlabel('Chapter')
plt.ylabel('Surprise occurrences')

# NLTK Vader
Valence Aware Dictionary for sEntiment Reasoning (aka Vader) blended a lexicon approach with some basic grammatical rules and syntactical conventions in order to express sentiment polarity and intensity.

It is currently also part of the nltk NLP package

In [ ]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()
scores = sia.polarity_scores('Yay')
scores


In [ ]:
# Compound usually paints the best picture of the overall sentiment of the text. It is obtained from combining negative, neutral and positive values.
# We will split the words into positive, neutral and negative based on the compound value

list_nltk_sentiments = ['positive' if sia.polarity_scores(word)['compound'] > 0 else 'negative' if sia.polarity_scores(word)['compound'] < 0 else 'neutral' for word in ss_df['word']]

In [ ]:
# Add the information to our dataframe
ss_df = ss_df.assign(nltk_sentiment = list_nltk_sentiments)
ss_df.head(500)[450:]


In [ ]:
# Now let’s try to do a similar analysis to the one we did with Afinn
# First we get rid of the neutral values and then count the positive and negative values per chapter

from siuba import *
test_df = count(filter(ss_df, _.nltk_sentiment != 'neutral'), 'chapter', 'nltk_sentiment')
test_df


In [ ]:
# Multiply by -1 the count values for the negative sentiments so as to be able to sum up the 2 sentiments
test_df.loc[test_df['nltk_sentiment'] == 'negative', 'n'] *= -1
test_df


In [ ]:
# Time to group the values by chapter and get the sum of each 2 values (positive and negative)

test_df_sentiment = test_df.groupby(test_df.chapter).sum().reset_index()
test_df_sentiment


In [ ]:
(ggplot(test_df_sentiment)          # defining what data to use
    + aes(x='chapter', y='n')       # defining what variable to use
    + geom_bar(stat='identity')     # defining the type of plot to use
)


### Exercise 4
1. **Find the 10 words that contribute positively the most to the overall book sentiment and the 10 words that contribute negatively the most to the overall book sentiment.**

2. **Visualize these words and their contributions with the help of graphs. 
Create a plot for top 10 positive words and another one for top 10 negative words. You can use any plotting package that you want.**

Make use of **nltk_sentiment** (positive and negative) and **the number of occurrences** of the word. 

The word that contributes positively the most will be the word that appears most often in the book and has nltk_sentiment = positive.
The word that contributes negatively the most will be the word that appears most often in the book and has nltk_sentiment = negative.



In [ ]:
# 10 words that contribute positively the most to the overall book sentiment
ss_df_words = count(ss_df, 'word', 'word_score', 'nltk_sentiment')
# filter out words with word_score 0
ss_df_words = ss_df_words[ss_df_words['word_score'] != 0]
ss_df_words['total'] = ss_df_words['n'] * ss_df_words['word_score']

In [ ]:
# top positive words
ss_df_top_words_pos = ss_df_words.sort_values(by='total', ascending=False)
ss_df_top_words_pos.head(n=10)

In [ ]:
# top negative words
ss_df_top_words_neg = ss_df_words.sort_values(by='total', ascending=True)
ss_df_top_words_neg.head(n=10)

# Word Clouds

Word clouds are a beautiful, fun and useful way of representing and viewing text data. We’ve probably all seen one before. The basic concept is that of showing the most frequently appearing words in a piece of text and to scale these words proportionally to their frequency of appearance (e.g. a word appearing 500 times will have a higher font size and will be more obvious in the word cloud than a word appearing 100 times).

In [ ]:
%%capture
%pip install wordcloud

In [ ]:
# Let’s create a word cloud from our Sense & Sensibility book
from wordcloud import WordCloud

wordcloud = WordCloud(background_color='white', width=1000, height=800).generate(sense_sensibility_text)

plt.figure(figsize = (6, 6), facecolor = None) 
plt.imshow(wordcloud, interpolation='bilinear')
plt.show()

In [ ]:
# WordCloud objects can take custom lists of stop words as input. They make use of a default stopwords list if you don't specify any though.
# If we were to create the same word cloud as before and give an empty stop words list we'd be reminded why we want to get rid of these words.

from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator

wordcloud = WordCloud(background_color='white', width=1000, height=800, stopwords=[]).generate(sense_sensibility_text)

plt.figure(figsize = (6, 6), facecolor = None) 
plt.imshow(wordcloud, interpolation='bilinear')
plt.show()

In [ ]:
# We can also save our word clouds to file
wordcloud.to_file("saved_wcloud.png")


In [ ]:
# We have the posibility of shaping the word cloud based on an image
import numpy as np
from PIL import Image

image_mask = np.array(Image.open("image_mask.png"))
wordcloud = WordCloud(background_color='white', mask=image_mask, width=2000, height=1800, contour_width=3, contour_color='steelblue').generate(sense_sensibility_text)

plt.figure(figsize = (16, 16), facecolor = None) 
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()


In [ ]:
# We have to specify a transparent .png image to serve as a mask in order to get our shapes.
# The png image needs to have pixel values = 255 rather than 0 in order to work.

# Here's an example of how to do that
import numpy as np
from PIL import Image

mask = np.array(Image.open("cat.png"))
mask[mask == 0] = 255
wordcloud = WordCloud(background_color='white', mask=mask, width=1000, height=900, contour_width=3, contour_color='steelblue').generate(sense_sensibility_text)

plt.figure(figsize = (16, 16), facecolor = None) 
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()


### Exercise 5
1. Create a word cloud from **all the positive words** in Sense and Sensibility. 

    Use colormap='Set2' to color this word cloud.
2. Create a word cloud from **all the negative words** in Sense and Sensibility.

Observations

*   Make use of the sentiment given by nltk
*   Use a custom image mask (or two, one for each) that's different from the 2 masks given in the lab.
*   Give colormap='Set2' as a parameter to WordCloud() when generating the positive word cloud 

Suggestions

*   World clouds also have a generate_from_frequencies() method of generating the cloud. This expects a dictionary of form {'word_a' : 20, 'word_b' : 3, 'word_c' : 120, etc.} as input where words _a, _b and _c are the words and 20, 3, 120 are their respective occurrence frequency.
*   Dataframes can be transformed into dictionaries - e.g. .to_dict()



In [ ]:
# positive words
ss_df_words_pos = ss_df_words[ss_df_words['nltk_sentiment'] == 'positive']
ss_df_words_pos.drop(['word_score', 'nltk_sentiment', 'total'], axis=1, inplace=True)
ss_df_words_pos.set_index('word', inplace=True)

words_pos_dict = ss_df_words_pos.to_dict()['n']

goomba_mask = np.array(Image.open("goomba.png"))
goomba_mask[goomba_mask == 0] = 255

wordcloud = WordCloud(background_color='white', width=1000, height=900, mask=goomba_mask, colormap='Set2',
                      contour_width=3, contour_color='magenta').generate_from_frequencies(words_pos_dict)

plt.figure(figsize=(16, 16), facecolor=None)
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()


In [ ]:
ss_df_words_neg = ss_df_words[ss_df_words['nltk_sentiment'] == 'negative']
ss_df_words_neg.drop(['word_score', 'nltk_sentiment', 'total'], axis=1, inplace=True)
ss_df_words_neg.set_index('word', inplace=True)

words_neg_dict = ss_df_words_neg.to_dict()['n']

psyduck_mask = np.array(Image.open("psyduck.png"))
psyduck_mask[psyduck_mask == 0] = 255

wordcloud = WordCloud(background_color='white', width=1000, height=900, mask=psyduck_mask, colormap='viridis',
                      contour_width=2, contour_color='steelblue').generate_from_frequencies(words_neg_dict)

plt.figure(figsize=(16, 16), facecolor=None)
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()


# Pretrained models (extra information)
*Machine Learning trained models are out of the scope of lab 3. This is purely informative. However, you can make use of any of these models in your exam project if you want better classifications.*

Machine Learning models - models that were trained on large datasets in order to predict something about the text data (e.g. whether a text is positive/negative) outclass their lexicon predecessors.

Models are usually trained on datasets with a well defined scope (such as movie reviews for instance). They may not be able to generalize properly for all types of text data, but they perform extremely well in the tasks they've been trained on.

If you use a model trained for movie review positivity prediction to try and classify positivity/negativity of an old book, you're likely going to have poor results. 
If you use the same model for other movie/book/hotel reviews, the model will most likely perform very well.


In [ ]:
%pip install transformers

In [ ]:
# Get our sentiment analysis model from pipeline
from transformers import pipeline
sentiment_pipeline = pipeline("sentiment-analysis")


In [ ]:
# Apply it to some of our text data
data = ["I do not love you", "I hate you", "It's been a rough situation", "It went terribly well"]
sentiment_pipeline(data) 
# We can see that the model is not naive at all
# We can combine positive and negative words without confusing it

You can visit https://huggingface.co/docs/transformers/main_classes/pipelines for plenty of other models.